# Stage 6: Code Generation with Expert Prompting

In this stage, we'll learn how to:
1. Load retrieved documentation and code examples from Stage 5
2. Inject context into expert-crafted prompts
3. Use structured output (Pydantic) with reasoning field
4. Generate Gurobi solver code using LLM

Key Concept: Context + Expert Prompt → Correct Code
- Context: Retrieved docs provide theoretical knowledge + implementation patterns
- Expert Prompt: Specifies requirements (function name, error handling, return type)
- Structured Output: Enforces consistency (code + reasoning_steps)
- Reasoning Field: Forces LLM to validate its own code before returning

Output: Generated Gurobi code for each problem

In [1]:
import pickle
import json
from pydantic import BaseModel, Field
import re

from config import (
    INCLUDE_REASONING_FIELD,
    VERBOSE,
    OPENROUTER_MODEL,
    LLM_TEMPERATURE
)
from llm_helper import OpenRouterLLM

## SECTION 1: EXPERT PROMPTING TEMPLATES

In [2]:
# System Prompt: Define LLM's role and context integration strategy
EXPERT_SYSTEM_PROMPT = """
You are an expert operations researcher and Python developer specializing in Gurobi-based linear programming, with deep expertise in translating complex word problems into precise mathematical models. Your knowledge encompasses both theoretical optimization principles and practical implementation techniques.

Context Integration Instructions:
1. Knowledge Context:
   * Analyze the following knowledge-based context snippets:
     {knowledge_context}
   * Extract relevant theoretical concepts, constraints, and modeling patterns
   * Apply domain-specific best practices and optimization techniques mentioned
   * Incorporate appropriate terminology and mathematical formulations

2. Code Example Context:
   * Study the following code examples for relevant implementation patterns:
     {code_context}
   * Adapt useful variable naming conventions and constraint structures
   * Reference similar problem-solving approaches when applicable
   * Maintain consistency with demonstrated Gurobi API usage patterns

Your role is to synthesize this contextual information with your expertise to generate optimal solutions.
"""

# User Prompt: Specify exact requirements for generated code
EXPERT_USER_PROMPT = """
Your primary task is to generate Python functions that solve linear programming problems using the Gurobi API, with a specific focus on integer-variable models. You must analyze both the context and problem descriptions meticulously to ensure every aspect is properly represented in both the mathematical model and code implementation.

Technical Requirements:
1. Create a Python function named ```solve``` that:
   * Declares ALL variables mentioned in the problem
   * Implements every constraint and objective function exactly as described
   * Provides comprehensive error handling, returning None for any exceptions
   * Returns only the numerical objective value (float) upon successful solving

Critical Implementation Rules:
1. Variables:
   * Must be strictly based on problem description quantities
   * Use contextually meaningful variable names
2. Constraints:
   * Every inequality/equality must be explicitly modeled
   * No assumptions beyond problem description
   * Clear, meaningful constraint names
3. Error Handling:
   * Return None for any exceptions (syntax errors, infeasibility, etc.)
4. Code Quality:
   * Professional-grade implementation
   * Proper Gurobi API syntax
   * No external dependencies beyond Gurobi
   * No console output during execution
   * No placeholder or incomplete implementations
   * No undefined variables or constraints

Required Output:
1. Complete Python code implementation:
   * Contains all necessary Gurobi import statements
   * Uses proper error handling with try-except blocks
   * Guarantees resource cleanup
   * The complete implementation along with return statement must be inside the function called ```solve```

2. Reasoning steps explaining:
   * How each problem element maps to specific code components
   * Verification that all variables are defined
   * Confirmation of complete constraint coverage
   * Proof that no aspect of the problem description is omitted

Your code must be directly executable with the Gurobi API, handle all error cases appropriately, and perfectly represent the mathematical model described in the problem. Any deviation from these requirements or omission of problem elements will be considered a failure.

Problem Description:
{question}
"""

## SECTION 2: STRUCTURED OUTPUT SCHEMA

In [3]:
if INCLUDE_REASONING_FIELD:
    class GurobiSolution(BaseModel):
        """Schema for LLM output with reasoning."""
        code: str = Field(description="Complete Python code using Gurobi API that solves the LP problem")
        reasoning_steps: str = Field(description="Technical justification mapping code components to problem requirements")
else:
    class GurobiSolution(BaseModel):
        """Schema for LLM output without reasoning (ablation study)."""
        code: str = Field(description="Complete Python code using Gurobi API that solves the LP problem")

## SECTION 3: CODE GENERATION FUNCTION

In [4]:
def generate_gurobi_solution(problem_description: str,
                             knowledge_context: str,
                             code_context: str) -> GurobiSolution:
    """
    Generate Gurobi solver code using LLM with structured output.

    Args:
        problem_description: Natural language LP problem
        knowledge_context: Retrieved documentation snippets
        code_context: Retrieved code examples

    Returns:
        GurobiSolution object (code + optional reasoning)

    Why Structured Output:
        Without schema enforcement, LLMs return inconsistent formats:
        - Sometimes code wrapped in markdown ```python blocks
        - Sometimes code + explanation mixed together
        - Sometimes just explanation with no code

        Pydantic schema forces consistent JSON output we can reliably parse.

    Why Reasoning Field:
        Forcing the LLM to explain its code improves accuracy:
        - Acts like chain-of-thought prompting for code
        - LLM catches its own mistakes while explaining
        - We get free documentation of the generated code
    """
    try:
        # Format prompts with context
        system_prompt = EXPERT_SYSTEM_PROMPT.format(
            knowledge_context=knowledge_context,
            code_context=code_context
        )

        user_prompt = EXPERT_USER_PROMPT.format(
            question=problem_description
        )

        # Add JSON schema instruction to user prompt
        schema_instruction = f"\n\nYou must respond with valid JSON matching this schema:\n{json.dumps(GurobiSolution.model_json_schema(), indent=2)}"
        user_prompt_with_schema = user_prompt + schema_instruction

        # Initialize LLM
        llm = OpenRouterLLM()

        # Call LLM with structured output
        response = llm.client.chat.completions.create(
            model=OPENROUTER_MODEL,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt_with_schema}
            ],
            temperature=llm.temperature,
            response_format={"type": "json_object"}  # ← Enforce JSON mode
        )

        # Parse and validate response
        json_content = response.choices[0].message.content

        # Strip markdown code blocks if present
        json_content = json_content.strip()
        if json_content.startswith('```json'):
            json_content = json_content[7:]  # Remove ```json
        if json_content.startswith('```'):
            json_content = json_content[3:]  # Remove ```
        if json_content.endswith('```'):
            json_content = json_content[:-3]  # Remove trailing ```
        json_content = json_content.strip()

        solution = GurobiSolution.model_validate_json(json_content)
        return solution

    except Exception as e:
        print(f"   ❌ Error in code generation: {str(e)}")
        return None


def extract_code(code_string: str) -> str:
    """
    Extract Python code from string (handles markdown blocks).

    Args:
        code_string: Raw code string (may contain ```python blocks)

    Returns:
        Clean Python code
    """
    # Try to extract from markdown code blocks
    code_pattern = re.findall(r"```python\n(.*?)\n```", code_string, re.DOTALL)

    if code_pattern:
        return "\n".join(code_pattern)
    else:
        # No markdown, return as-is
        return code_string

## SECTION 4: EXECUTE PIPELINE

In [5]:
print("=" * 80)
print("STAGE 6: CODE GENERATION WITH EXPERT PROMPTING")
print("=" * 80)

# Initialize LLM
print(f"\nLLM Configuration:")
print(f"   Model: {OPENROUTER_MODEL}")
print(f"   Temperature: {LLM_TEMPERATURE}")
print(f"   Reasoning Field: {'Enabled' if INCLUDE_REASONING_FIELD else 'Disabled'}")

# Load retrieval results from Stage 5
print(f"\nLoading retrieval results from Stage 5...")
with open('tutorial_retrievals.pkl', 'rb') as f:
    retrieval_results = pickle.load(f)

print(f"   Loaded retrievals for {len(retrieval_results)} problems\n")

# Store generation results
generation_results = []

# Process each problem
print(f"Generating code for {len(retrieval_results)} problems...")
print("   (This may take 10-15 minutes)\n")

for i, result in enumerate(retrieval_results, 1):
    print(f"{'─' * 80}")
    print(f"Problem #{result['problem_id']} ({i}/{len(retrieval_results)})")
    print(f"{'─' * 80}")

    # Extract problem info
    question = result['question']
    keywords = result['keywords']

    # Format contexts
    # Documentation context: Join page_content from top-2 docs
    doc_context = '\n\n'.join([
        doc.page_content for doc in result['doc_retrievals']
    ])

    # Code context: Join original_text from top-2 code examples
    code_context = '\n\n'.join([
        doc.metadata['original_text'] for doc in result['code_retrievals']
    ])

    print(f"   Keywords: {keywords}")
    print(f"   Doc context: {len(doc_context)} chars")
    print(f"   Code context: {len(code_context)} chars")

    # Generate code
    print(f"   Generating code with {OPENROUTER_MODEL}...")
    solution = generate_gurobi_solution(question, doc_context, code_context)

    if solution:
        print(f"   Code generated successfully")

        generation_results.append({
            'problem_id': result['problem_id'],
            'question': question,
            'keywords': keywords,
            'expected_objective': result['expected_objective'],
            'generated_code': extract_code(solution.code),
            'reasoning': solution.reasoning_steps if INCLUDE_REASONING_FIELD else None,
            'doc_context': doc_context,
            'code_context': code_context
        })
    else:
        print(f"   Code generation failed")
        generation_results.append({
            'problem_id': result['problem_id'],
            'question': question,
            'keywords': keywords,
            'expected_objective': result['expected_objective'],
            'generated_code': None,
            'reasoning': None,
            'doc_context': doc_context,
            'code_context': code_context
        })

    print()

STAGE 6: CODE GENERATION WITH EXPERT PROMPTING

LLM Configuration:
   Model: google/gemini-2.5-flash
   Temperature: 0.0
   Reasoning Field: Enabled

Loading retrieval results from Stage 5...
   Loaded retrievals for 2 problems

Generating code for 2 problems...
   (This may take 10-15 minutes)

────────────────────────────────────────────────────────────────────────────────
Problem #0 (1/2)
────────────────────────────────────────────────────────────────────────────────
   Keywords: Fishery Transportation, Maximize Fish, Budget Constraint, Trip Constraints, Continuous Variables, Gurobi Model, setObjective
   Doc context: 3406 chars
   Code context: 2619 chars
   Generating code with google/gemini-2.5-flash...
   Code generated successfully

────────────────────────────────────────────────────────────────────────────────
Problem #1 (2/2)
────────────────────────────────────────────────────────────────────────────────
   Keywords: Printer Production, Profit Maximization, Production Capa

In [6]:
print(f"{'=' * 80}")
print("DEMONSTRATION: Generated Code Examples")
print(f"{'=' * 80}\n")

for result in generation_results:
    print(f"{'─' * 80}")
    print(f"Problem #{result['problem_id']}")
    print(f"{'─' * 80}\n")

    # Show problem (truncated)
    problem_text = result['question']
    if len(problem_text) > 200:
        problem_text = problem_text[:200] + "..."

    print("Problem:")
    print(f"   {problem_text}\n")

    # Show generated code
    if result['generated_code']:
        print("Generated Code:")
        print(f"{'┌' + '─' * 78 + '┐'}")

        # Print code with line numbers
        code_lines = result['generated_code'].split('\n')
        for i, line in enumerate(code_lines[:30], 1):  # Show first 30 lines
            print(f"│ {i:2d} | {line}")

        if len(code_lines) > 30:
            print(f"│    | ... ({len(code_lines) - 30} more lines)")

        print(f"{'└' + '─' * 78 + '┘'}\n")

        # Show reasoning if available
        if result['reasoning']:
            print("Reasoning Steps:")
            reasoning_lines = result['reasoning'].split('\n')
            for line in reasoning_lines:
                print(f"   {line}")
            print()

    else:
        print("   Code generation failed for this problem\n")

    print()

# Save results for Stage 7
print(f"Saving generation results for Stage 7...")
with open('tutorial_generated_code.pkl', 'wb') as f:
    pickle.dump(generation_results, f)

DEMONSTRATION: Generated Code Examples

────────────────────────────────────────────────────────────────────────────────
Problem #0
────────────────────────────────────────────────────────────────────────────────

Problem:
   A fishery wants to transport their catch. They can either use local sled dogs or trucks. Local sled dogs can take 100 fish per trip while trucks can take 300 fish per trip. The cost per trip for sled ...

Generated Code:
┌──────────────────────────────────────────────────────────────────────────────┐
│  1 | import gurobipy as gp
│  2 | from gurobipy import GRB
│  3 | 
│  4 | def solve():
│  5 |     try:
│  6 |         # Create a new model
│  7 |         m = gp.Model("fishery_transport")
│  8 | 
│  9 |         # 1. Declare variables
│ 10 |         # Number of trips for sled dogs (integer variable)
│ 11 |         sled_dog_trips = m.addVar(vtype=GRB.INTEGER, name="sled_dog_trips")
│ 12 |         # Number of trips for trucks (integer variable)
│ 13 |         truck_tri